## 환경 설정 

In [1]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch


# 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cuda


## 시드고정

In [2]:
# 시드 고정
def seed_everything(seed=47):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## instructBLIP 모델 구조 커스텀

In [8]:
import torch
import torch.nn as nn
from transformers import (
    InstructBlipForConditionalGeneration, 
    T5ForConditionalGeneration,
    InstructBlipConfig,
    InstructBlipProcessor, 
    InstructBlipVisionConfig, 
    InstructBlipQFormerConfig, 
    T5Config
)


# 1. 기존 모델 config 로드
vision_config = InstructBlipVisionConfig.from_pretrained("Salesforce/instructblip-flan-t5-xl")
qformer_config = InstructBlipQFormerConfig.from_pretrained("Salesforce/instructblip-flan-t5-xl")
text_config = T5Config.from_pretrained("google/flan-t5-large")

# T5 config 조정 (InstructBLIP에 맞게)
text_config.is_encoder_decoder = True
text_config.use_cache = True
if not hasattr(text_config, 'bos_token_id') or text_config.bos_token_id is None:
    text_config.bos_token_id = 1  # T5의 경우 보통 0

# 2. 새로운 config로 InstructBLIP 모델 생성
config = InstructBlipConfig.from_vision_qformer_text_configs(
    vision_config, qformer_config, text_config
)

# 3. 빈 모델 초기화
model = InstructBlipForConditionalGeneration(config)

print("모델 구조 초기화 완료")
print(f"Vision model hidden size: {config.vision_config.hidden_size}")
print(f"Q-Former hidden size: {config.qformer_config.hidden_size}")
print(f"Text model hidden size: {config.text_config.hidden_size}")

# 4. 사전 훈련된 컴포넌트들 로드
print("사전 훈련된 가중치 로드 중...")

# 전체 원본 InstructBLIP 모델 로드 (vision과 qformer 가중치 추출용)
original_model = InstructBlipForConditionalGeneration.from_pretrained("Salesforce/instructblip-flan-t5-xl")

# Vision model 가중치 복사
model.vision_model.load_state_dict(original_model.vision_model.state_dict())
print("✓ Vision model 가중치 로드 완료")

# Q-Former 가중치 복사
model.qformer.load_state_dict(original_model.qformer.state_dict())
print("✓ Q-Former 가중치 로드 완료")

# Language model 로드 (flan-t5-large)
new_lm = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
model.language_model.load_state_dict(new_lm.state_dict())
print("✓ Language model (flan-t5-large) 가중치 로드 완료")

# 메모리 정리
del original_model, new_lm
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# 5. Language projection layer 재초기화
# Q-Former의 출력을 새로운 language model의 입력 차원에 맞게 조정
qformer_hidden_size = config.qformer_config.hidden_size  # 768
text_hidden_size = config.text_config.hidden_size        # 1024 (flan-t5-large)

# 기존 projection layer를 새로운 차원에 맞게 교체
model.language_projection = nn.Linear(
    qformer_hidden_size, 
    text_hidden_size, 
    bias=True
)

# Xavier uniform 초기화로 안정적인 학습 시작
nn.init.xavier_uniform_(model.language_projection.weight)
print(f"✓ Language projection layer 재초기화 완료: {qformer_hidden_size} -> {text_hidden_size}")

# 6. 모델 검증
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n모델 정보:")
print(f"총 파라미터 수: {total_params:,}")

모델 구조 초기화 완료
Vision model hidden size: 1408
Q-Former hidden size: 768
Text model hidden size: 1024
사전 훈련된 가중치 로드 중...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.71it/s]


✓ Vision model 가중치 로드 완료
✓ Q-Former 가중치 로드 완료
✓ Language model (flan-t5-large) 가중치 로드 완료


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ Language projection layer 재초기화 완료: 768 -> 1024

모델 정보:
총 파라미터 수: 1,955,574,528
훈련 가능한 파라미터 수: 1,955,574,528


## 토크나이저 계승

In [9]:
from transformers import T5Tokenizer

# 기본적으로 기존 InstructBLIP processor 사용
# tokenizer는 자동으로 flan-t5-large에 맞게 조정됨
processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-flan-t5-xl")
xl_tokens = set(processor.tokenizer.get_vocab().keys())     # 원래 토크나이저에서 vocab 추출

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
large_tokens = set(tokenizer.get_vocab().keys())     # 바꿀 토크나이저가 원래 갖고있던 vocab 추출
tokens_to_add = list(xl_tokens - large_tokens)      # 추가로 계승할 토큰
num_added = tokenizer.add_tokens(tokens_to_add)   # 3) flan‑t5‑large 토크나이저에 추가
print(f"Added {num_added} tokens from flan‑t5‑xl into flan‑t5‑large tokenizer")

processor.tokenizer = tokenizer
model.resize_token_embeddings(len(tokenizer))  # LM head(임베딩) 크기 조정
model.language_model.resize_token_embeddings(len(tokenizer))
model.config.text_config.vocab_size = len(tokenizer)
model.config.vocab_size = len(tokenizer)
del tokenizer   # 메모리 정리

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Added 2 tokens from flan‑t5‑xl into flan‑t5‑large tokenizer


## 더미입력으로 확인

In [ ]:
# 더미입력으로 확인
try:
    from PIL import Image
    import numpy as np
    import requests
    
    # PIL 이미지로 더미 이미지 생성 (RGB, 224x224)
    # dummy_image_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    # dummy_image = Image.fromarray(dummy_image_array, mode='RGB')
    url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
    dummy_image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    dummy_text = "Describe this image."
    
    print("더미 이미지 생성 완료")
    print(f"이미지 크기: {dummy_image.size}, 모드: {dummy_image.mode}")
    
    # 입력 전처리
    inputs = processor(
        images=dummy_image, 
        text=dummy_text, 
        return_tensors="pt"
    )
    
    print("입력 전처리 완료")
    print(f"입력 텐서 키: {list(inputs.keys())}")
    print(f"이미지 텐서 shape: {inputs['pixel_values'].shape}")
    print(f"텍스트 토큰 shape: {inputs['input_ids'].shape}")
    
    # T5 모델을 위한 decoder_input_ids 생성
    # T5는 decoder 시작 시 pad_token_id를 사용
    batch_size = inputs['input_ids'].shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1), 
        processor.tokenizer.pad_token_id, 
        dtype=torch.long
    )
    inputs['decoder_input_ids'] = decoder_input_ids
    
    print(f"decoder_input_ids 추가: {decoder_input_ids.shape}")
    
    # Forward pass 테스트 (training mode)
    model.train()  # training mode로 설정
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("✓ Forward pass 성공")
    print(f"출력 logits shape: {outputs.logits.shape}")
    print(f"출력 logits 범위: [{outputs.logits.min().item():.4f}, {outputs.logits.max().item():.4f}]")
    
    # Generation 테스트도 수행
    print("\nGeneration 테스트...")
    model.eval()  # evaluation mode로 설정
    with torch.no_grad():
        # decoder_input_ids 제거 (generate에서는 자동 생성)
        gen_inputs = {k: v for k, v in inputs.items() if k != 'decoder_input_ids'}
        generated_ids = model.generate(
            **gen_inputs,
            max_new_tokens=30,
            do_sample=True,
            num_beams=3,
            repetition_penalty=1.5,
        )
    
    # 생성된 텍스트 디코딩
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"✓ Generation 성공")
    print(f"생성된 텍스트 (샘플): '{generated_text}...'")
      
    
except Exception as e:
    print(f"✗ 테스트 실패: {str(e)}")
    # 에러 발생 시 더 자세한 디버깅 정보 제공
    import traceback
    print("상세 에러 정보:")
    traceback.print_exc()


print("\n모델 개조 및 테스트 완료!")

# ★★ Finetuning ★★  

## PEFT-LoRA 설정

In [ ]:
# PEFT-LoRA 적용
from peft import LoraConfig, get_peft_model
from torch import nn

target_module_names = []
for name, module in model.named_modules():
    if ("qformer" in name or "language_model" in name) and isinstance(module, (nn.Linear, nn.Embedding)):   # Qformer와 langauge모델만 훈련
        target_module_names.append(name.split('.')[-1]) 

# 중복 제거
target_module_names = list(set(target_module_names))

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=target_module_names        
)
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

trainable params: 24,708,720 || all params: 1,980,230,000 || trainable%: 1.2478
None


In [6]:
# A-OKVQA 데이터셋
from datasets import load_dataset

train_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="train")
val_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="validation")

a = len(train_ds)
# 데이터 유효성 확인
train_ds = train_ds.filter(lambda example: len(example['choices']) == 4)
train_ds = train_ds.filter(lambda example: example['correct_choice_idx'] in [0,1,2,3])
print(f"필터링된 샘플 갯수: {len(train_ds) - a}")

필터링된 샘플 갯수: 0


In [13]:
# 커스텀 데이터셋
class AokvqaDataset(torch.utils.data.Dataset):
    """A-OKVQA dataset."""

    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        answer_idx = sample['correct_choice_idx']
        answer = sample['choices'][answer_idx]
        image = sample['image'].convert('RGB')
        prompt = f"""
        <image>
        Based on the image, choose the correct option to the following question.

        Question: {sample['question']}

        Options:
        A {sample['choices'][0]}
        B {sample['choices'][1]}
        C {sample['choices'][2]}
        D {sample['choices'][3]}

        Answer:
        """
        encoding = self.processor(image, prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt")
        
        # remove batch dimension
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        label_encoding = self.processor.tokenizer(
            answer,
            max_length=128, # 라벨의 최대 길이 (충분히 크게)
            padding="max_length", # 여기서는 max_length 패딩이 적절
            truncation=True,
            return_tensors="pt"
        )
        encoding["labels"] = label_encoding.input_ids.squeeze(0)
        encoding["labels"][encoding["labels"] == self.processor.tokenizer.pad_token_id] = -100
        return encoding
        # InstructBLIP은 일반적으로 답을 프롬프트에 포함하여 훈련합니다.
        # InstructBLIPProcessor는 이미지를 pixel_values로, 텍스트를 input_ids, attention_mask 등으로 변환합니다.
        # IMPORTANT: 'labels'는 모델의 타겟 아웃풋이므로, 이 프롬프트 자체를 인코딩할 때 포함시키지 않습니다.
        # labels는 별도로 처리해야 합니다.
        # 이 부분에서 `text`에는 질문과 옵션, 그리고 "Answer:"까지 포함된 프롬프트를 전달합니다.
        # InstructBLIPForConditionalGeneration의 forward 메서드는 `labels` 인자를 받습니다.
        # 이 `labels`는 모델이 생성해야 할 "정답" 부분에 해당합니다.
        # 따라서 `processor`를 호출할 때는 `text`에 질문과 옵션만 주고, `labels`를 별도로 생성해야 합니다.
        # InstructBLIP은 보통 <image> 토큰을 사용하고, Answer: 뒤에 LLM이 생성할 텍스트를 기대합니다.
        # 따라서 `text`에는 Answer: 앞부분까지의 프롬프트를 주고, `labels`에는 Answer: 뒷부분의 정답을 토큰화하여 제공해야 합니다.
        # VQA 태스크의 경우, InstructBLIP의 일반적인 훈련 방식은 "Question: <question_text> Answer: <answer_text>"입니다.
        # Multiple Choice의 경우, "Question: <question_text> Options: <options_text> Answer: <answer_text>"가 될 수 있습니다.
        # `processor`의 `text` 인자에 전체 프롬프트 (정답 포함)를 넣어주고, `labels`는 나중에 콜레이트 함수에서 처리합니다.
        
        # InstructBLIP의 기본적인 VQA Fine-tuning 예시를 참고하면,
        # prompt = f"Question: {sample['question']} Answer:"
        # answer_text = sample['choices'][sample['correct_choice_idx']]
        
        # VQA with Multiple Choice:
        # processor는 `pixel_values`, `qformer_input_ids`, `qformer_attention_mask`, `input_ids`, `attention_mask`를 생성합니다.
        # 여기서 `input_ids`와 `attention_mask`는 LLM에 들어갈 전체 프롬프트 (Q-Former 출력 삽입 자리 포함) 입니다.

        # 'labels'는 모델이 생성해야 할 '정답' 텍스트를 토큰화한 것입니다.
        # InstructBlipProcessor는 텍스트를 인코딩할 때 이미 LLM의 토크나이저를 사용하므로,
        # labels도 동일한 토크나이저로 인코딩해야 합니다.
        
        # `labels`는 `generate`에서 사용되는 것이 아니라, `forward` 메서드의 타겟으로 사용됩니다.
        # 모델은 `input_ids`를 입력받아 `labels`와 비교하여 손실을 계산합니다.
        # 따라서 `labels`는 LLM이 "Answer: " 이후에 생성해야 할 정답 텍스트를 토큰화한 것이어야 합니다.
        
        # AokvqaDataset.__getitem__에서는 `labels`를 정답 텍스트를 인코딩한 형태로 저장합니다.
        # 실제 모델 훈련 시에는 이 `labels`가 LLM의 출력과 비교됩니다.
    

def collate_fn(batch):
    # 'pixel_values', 'qformer_input_ids', 'qformer_attention_mask', 'input_ids', 'attention_mask', 'labels'
    # 이 모든 키들이 배치로 쌓여야 합니다.
    processed_batch = {}
    for key in batch[0].keys():
        processed_batch[key] = torch.stack([sample[key] for sample in batch])
        
    return processed_batch
    
    
train_ds = AokvqaDataset(dataset=train_ds, processor=processor)
val_ds = AokvqaDataset(dataset=val_ds, processor=processor)

NameError: name 'train_ds' is not defined

In [8]:
from torch.utils.data import DataLoader

batch_size = 2
train_dataloader = DataLoader(train_ds, batch_size=batch_size, collate_fn = collate_fn, shuffle=True, pin_memory=True, num_workers=4)
valid_dataloader = DataLoader(val_ds, batch_size=batch_size, collate_fn = collate_fn, pin_memory=True, num_workers=4)

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9, last_epoch=-1)

In [10]:
model.to(device)

PeftModel(
  (base_model): LoraModel(
    (model): InstructBlipForConditionalGeneration(
      (vision_model): InstructBlipVisionModel(
        (embeddings): InstructBlipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
        )
        (encoder): InstructBlipEncoder(
          (layers): ModuleList(
            (0-38): 39 x InstructBlipEncoderLayer(
              (self_attn): InstructBlipAttention(
                (qkv): Linear(in_features=1408, out_features=4224, bias=True)
                (projection): Linear(in_features=1408, out_features=1408, bias=True)
              )
              (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
              (mlp): InstructBlipMLP(
                (activation_fn): GELUActivation()
                (fc1): Linear(in_features=1408, out_features=6144, bias=True)
                (fc2): Linear(in_features=6144, out_features=1408, bias=True)
              )
              (layer_n

In [ ]:
# --- 훈련 루프 ---
num_epochs = 5
patience = 10
min_eval_loss = float("inf")
early_stopping_hook = 0
tracking_information = []
gradient_accumulation_steps = 128 // batch_size

for epoch in range(num_epochs):
    # ========================= Training =========================
    model.train()
    
    step = 1
    epoch_loss = 0
    
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]"):
        pixel_values = batch.pop("pixel_values").to(device)
        input_ids = batch.pop("input_ids").to(device)
        qformer_input_ids = batch.pop('qformer_input_ids').to(device)
        qformer_attention_mask = batch.pop('qformer_attention_mask').to(device)
        labels = batch.pop('labels').to(device)

        outputs = model(
            input_ids=input_ids,
            pixel_values=pixel_values,
            qformer_input_ids=qformer_input_ids,
            qformer_attention_mask=qformer_attention_mask,
            labels=labels
            )
        loss = outputs.loss
        loss /= gradient_accumulation_steps

        # 역전파
        loss.backward()
        
        if step%gradient_accumulation_steps == 0 or step == len(train_dataloader):
            optimizer.step()
            optimizer.zero_grad()
        
        epoch_loss += loss.item() * gradient_accumulation_steps
        step += 1

    # ========================= Validation =========================
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in tqdm(valid_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Validation]"):

            pixel_values = batch.pop("pixel_values").to(device)
            input_ids = batch.pop("input_ids").to(device)
            qformer_input_ids = batch.pop('qformer_input_ids').to(device)
            qformer_attention_mask = batch.pop('qformer_attention_mask').to(device)
            labels = batch.pop('labels').to(device)
    
            outputs = model(
                input_ids=input_ids,
                pixel_values=pixel_values,
                qformer_input_ids=qformer_input_ids,
                qformer_attention_mask=qformer_attention_mask,
                labels=labels
                )
            loss = outputs.loss
            
            eval_loss += loss.item()

    # --- 에포크 마무리 및 로깅 ---
    avg_train_loss = epoch_loss / len(train_dataloader)
    avg_eval_loss = eval_loss / len(valid_dataloader)
    current_lr = optimizer.param_groups[0]["lr"]

    tracking_information.append((avg_train_loss, avg_eval_loss, current_lr))
    print(f"Epoch: {epoch+1} | Train Loss: {avg_train_loss:.4f} | Eval Loss: {avg_eval_loss:.4f} | LR: {current_lr}")
    
    # 스케줄러 업데이트
    scheduler.step()

    # 조기 종료 및 모델 저장
    # Early stopping은 전체 loss 합이 아닌 평균 loss로 비교하는 것이 더 직관적입니다.
    if avg_eval_loss < min_eval_loss:
        min_eval_loss = avg_eval_loss
        early_stopping_hook = 0
        model.save_pretrained("Model/instructblip-lora")
        model.base_model.save_pretrained('Model/instructblip-base')
        processor.save_pretrained("Model/instructblip-processor")
        print(f"Validation loss decreased ({min_eval_loss:.4f}). Saving model...")
    else:
        early_stopping_hook += 1
        if early_stopping_hook >= patience:
            print(f"Early stopping at epoch {epoch+1} as validation loss did not improve for {patience} epochs.")
            break

Epoch 1/5 [Training]:   0%|          | 0/8528 [00:00<?, ?it/s]Expanding inputs for image tokens in InstructBLIP should be done in processing. Please follow instruction here (https://gist.github.com/zucchini-nlp/e9f20b054fa322f84ac9311d9ab67042) to update your InstructBLIP model. Using processors without these attributes in the config is deprecated and will throw an error in v4.50.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Epoch 1/5 [Training]:  10%|▉         | 825/8528 [06:37<1:02:18,  2.06it/s]

## Inference sample

In [ ]:
import torch
from PIL import Image
import requests

url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

question = "What is unusual about this image?"
choices = [
    "There is a dog driving a car.",
    "The sky is green and the grass is blue.",
    "The man is hanging from the back of the car and ironing his clothes",
    "There are two moons in the sky."
]

# 🔹 Instruction-style prompt 생성
prompt = (
    "<image>\n"
    "Based on the image, choose the correct option to the following question.\n"
    f"Question: {question}\n"
    "Options:\n" +
    "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
    "\nAnswer:"
)

# 🔹 전처리
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

# 🔹 추론
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        num_beams=5,
        repetition_penalty=1.5,
        top_p = 0.9,
    )

# 🔹 결과 디코딩
answer_text = processor.tokenizer.batch_decode(out_ids, skip_special_tokens=True)[0].strip()
print(f"Predicted Answer: {answer_text}")
Image._show(image)

In [ ]:
print(model.language_model)

## Submission

In [ ]:
# 추론
test = pd.read_csv('./test.csv')
results = []

# 정답 알파벳 추출 함수
def extract_answer_letter(text):
    match = re.search(r"\s*([A-Da-d])\b", text)
    return match.group(1).upper() if match else "?"


for _, row in tqdm(test.iterrows(), total=len(test)):
    image = Image.open(row['img_path']).convert("RGB")
    choices = [row[c] for c in ['A', 'B', 'C', 'D']]

    prompt = (
        "<image>\n"
        "Based on the image, choose the correct option to the following question.\n"
        f"Question: {row['Question']}\n"
        "Options:"
        + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
        "\nAnswer:"
    )

    inputs = processor(images=image, text=prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt").to(device)

    output = model.generate(**inputs, 
                            num_beams=5, 
                            top_p=0.9, 
                            repetition_penalty=1.5, 
                            length_penalty=1.0, 
                            temperature=0.7, 
                            max_new_tokens=3, 
                            do_sample=False)
    decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
    print(decoded)
    results.append(extract_answer_letter(decoded))

print('✅ Done.')

In [ ]:
submission = pd.read_csv('./sample_submission.csv')
submission['answer'] = results
submission.to_csv('./baseline_submit.csv', index=False)
print("Done.")

## 모델 로드

In [ ]:
# 모델 로드
from transformers import InstructBlipForConditionalGeneration, AutoProcessor
from peft import PeftModel, PeftConfig

peft_model_path = "./Model/instructblip-lora"
config = PeftConfig.from_pretrained(peft_model_path)

model_c = InstructBlipForConditionalGeneration.from_pretrained('./Model/instructblip-base', device_map="auto")
model_c = PeftModel.from_pretrained(model_c, peft_model_path)

processor_c = AutoProcessor.from_pretrained("./Model/instructblip-base-processor")

In [21]:
# 데이컨 데이터셋
dacon_ds = load_dataset("csv",  data_files="./train.csv", split='train')

class DaconDataset(torch.utils.data.Dataset):
    """Official dataset from DACON."""

    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        image = Image.open(sample['img_path']).convert("RGB")
        answer = sample[sample['answer'].strip()]
        
        prompt = f"""
        <image>
        Based on the image, choose the correct option to the following question.

        Question: {sample['Question']}

        Options:
        A {sample['A']}
        B {sample['B']}
        C {sample['C']}
        D {sample['D']}

        Answer:
        """
        encoding = self.processor(image, prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt")
        
        # remove batch dimension
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        label_encoding = self.processor.tokenizer(
            answer,
            max_length=128, # 라벨의 최대 길이 (충분히 크게)
            padding="max_length", # 여기서는 max_length 패딩이 적절
            truncation=True,
            return_tensors="pt"
        )
        encoding["labels"] = label_encoding.input_ids.squeeze(0)
        encoding["labels"][encoding["labels"] == self.processor.tokenizer.pad_token_id] = -100
        return encoding


dacon_ds = DaconDataset(dataset=dacon_ds, processor=processor)
dacon_dataloader = DataLoader(dacon_ds, batch_size=batch_size, collate_fn = collate_fn, shuffle=True, pin_memory=True, num_workers=4)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

{'input_ids': tensor([[32101, 32101, 32101,  ...,     0,     0,     0],
        [32101, 32101, 32101,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'qformer_input_ids': tensor([[ 101, 1026, 3746,  ...,    0,    0,    0],
        [ 101, 1026, 3746,  ...,    0,    0,    0]]), 'qformer_attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'pixel_values': tensor([[[[ 0.9522,  1.1128,  0.8355,  ...,  1.8573,  1.8573,  1.8573],
          [ 0.9814,  1.1128,  0.8501,  ...,  1.8573,  1.8573,  1.8573],
          [ 0.9960,  1.1274,  0.8647,  ...,  1.8573,  1.8573,  1.8573],
          ...,
          [ 0.3829,  0.3391,  0.3099,  ...,  0.9960,  1.0836,  1.1858],
          [ 0.2369,  0.2077,  0.2077,  ...,  1.2004,  1.2296,  1.2296],
          [ 0.2223,  0.2223,  0.1931,  ...,  1.1566,  1.2004,  1.2150]],

         [[ 1.0093,  1.2044,  0.8743,  ...,  1.9998,  1.9998,  1.9998],
          [ 1.0243, 